# BabyAGI

**Domain:** Agentic AI  ·  **runnable:** yes

A refresher on **BabyAGI** — Yohei Nakajima's ~140-line script (March 2023) that became the canonical example of a **task-driven autonomous agent**. You give it a single *objective* and one seed task; it then loops on its own — doing a task, inventing follow-up tasks from the result, re-prioritizing the list, and repeating — until the work runs out (or you stop it).

Where [[autogen]] and [[crewai]] orchestrate *several* agents talking to each other, BabyAGI is one agent that **manages its own to-do list**. It's less a production framework than the clearest possible illustration of the *plan → act → reflect → re-plan* loop that sits underneath almost every autonomous-agent system.

## 1. What & Why

**BabyAGI** is a tiny autonomous-agent loop: from an `OBJECTIVE` and an initial task it spins up a self-managing task queue. Each iteration it pulls the top task, *executes* it with an LLM (using relevant past results as context), stores the result in a memory store, asks the LLM to *create* new tasks implied by that result, and then *re-prioritizes* the whole queue toward the objective. Repeat until the queue is empty or a step cap is hit.

**The problem it solves (conceptually).** A single LLM call answers one prompt; it can't decompose a fuzzy goal into steps, pursue them, and adapt as it learns. BabyAGI shows the smallest machinery that turns "here's a goal" into "here's an agent that keeps generating and working its own tasks." It made the *task-list-as-agent-memory* pattern legible to everyone.

**When to reach for it.** As a **learning tool / starting point** for understanding autonomous agents, and as a quick scaffold for open-ended, exploratory goals where you don't know the steps up front ("research X and draft a brief"). **When not to:** anything production-facing. It has no real stopping criterion, no cost ceiling, weak grounding, and no tool ecosystem out of the box. For real work, reach for a framework with state, tools, and guardrails — [[langgraph]], [[crewai]], or [[autogen]] — and keep BabyAGI as the mental model.

## 2. Mental Model

Think of BabyAGI as **an intern with a sticky-note to-do list and a notebook of everything they've learned.**

- The **objective** is the standing instruction ("write a brief on morning walks and focus").
- The **task queue** is the stack of sticky notes. The intern always works the top note.
- After finishing a note, they **jot the result in their notebook** (memory), then **write new sticky notes** for anything the result implies, and **re-order the pile** so the most objective-relevant note is on top.
- Before doing each note they **skim the notebook** for related past findings and use them as context.

That's the entire system — three LLM "skills" wrapped around a queue and a memory:

```
                 ┌──────────────── OBJECTIVE (fixed) ────────────────┐
                 │                                                    │
   ┌─────────────▼─────────────┐                                      │
   │  TASK QUEUE (deque)        │                                      │
   │  1. top task ──────────────┼──▶ EXECUTION AGENT ──▶ result ──▶ MEMORY (vector store)
   │  2. ...                    │        ▲                  │            │
   └─────────────▲──────────────┘        │ context          │            │
                 │                        └──────────────────┘            │
                 │                                                         │
   PRIORITIZATION AGENT ◀── new tasks ── TASK-CREATION AGENT ◀── result ───┘
   (reorder vs objective)                (invent follow-ups)
```

The loop is: **pop → execute (with retrieved context) → store → create tasks → re-prioritize → repeat.**

## 3. Key Concepts

| Concept | What it is |
|---|---|
| **Objective** | The single fixed goal string threaded into every agent prompt. Everything is judged relative to it. |
| **Task queue** | An ordered list (a `deque`) of pending tasks, each just `{"task_name": "..."}`. The agent always works the front. |
| **Execution agent** | LLM call that *performs* one task. Prompt = objective + task + retrieved context from memory. Returns a free-text result. |
| **Task-creation agent** | LLM call that reads the latest result and the remaining queue, then **invents new tasks** to append (de-duplicated against pending ones). This is what makes the agent open-ended. |
| **Prioritization agent** | LLM call that **re-orders** the whole queue so the most objective-relevant task is next. Prevents the list from drifting. |
| **Memory / context store** | Originally a **vector DB** (Pinecone, later Chroma/FAISS). Completed results are embedded and stored; before each task the agent retrieves the top-k most similar past results as context. |
| **The loop** | `while task_list: pop → execute → store → create → prioritize`. No built-in convergence test — it runs until empty or you cap iterations. |

> **Two BabyAGIs.** The **original** (2023) is the task-loop described here — the thing people mean by "BabyAGI." In **late 2024 Nakajima rebooted the project** as a *self-building* function framework ("functionz": a graph of registered functions with logging and a dashboard), which is a different beast. This notebook is about the original loop, since that's the durable idea; the repo's `classic/` folder still holds it.

## 4. Setup

The real BabyAGI needs an LLM and a vector store:

```bash
# Run the original script
git clone https://github.com/yoheinakajima/babyagi
cd babyagi && pip install -r requirements.txt
cp .env.example .env          # set OPENAI_API_KEY, OBJECTIVE, INITIAL_TASK
python babyagi.py             # classic/ folder in the current repo

# Or drive the same loop via LangChain's experimental wrapper
pip install langchain langchain-experimental langchain-openai faiss-cpu
export OPENAI_API_KEY=...
```

Requires **Python 3.9+** and an OpenAI (or compatible) key; the classic script also wants a vector store (Pinecone originally; Chroma/FAISS in later versions). There is **no offline default model** — every real run costs tokens and, crucially, *won't stop on its own*.

Examples 1 & 2 below are **pure standard-library** reimplementations of the BabyAGI loop and its vector-memory, so they run **offline with no key**. Example 3 shows the real `openai` / LangChain shapes and is gated behind an install + key check.

In [1]:
# Setup check — examples 1 & 2 need nothing; example 3 needs a library + key.
import importlib.util, os

has_openai = importlib.util.find_spec("openai") is not None
has_lc     = importlib.util.find_spec("langchain_experimental") is not None
has_key    = bool(os.getenv("OPENAI_API_KEY"))

print("openai installed:                ", has_openai)
print("langchain_experimental installed:", has_lc)
print("OPENAI_API_KEY present:          ", has_key)
print("\nExamples 1 & 2 run regardless (stdlib only).")

openai installed:                 True
langchain_experimental installed: False
OPENAI_API_KEY present:           False

Examples 1 & 2 run regardless (stdlib only).


## 5. Worked Examples

### Example 1 — The BabyAGI loop, from scratch

The whole system is three "agents" (just functions) wrapped around a task queue and a memory store. Below we reimplement the loop with **deterministic fake LLMs** so it runs offline and reproducibly — but the control flow (*pop → execute → store → create → prioritize*) is exactly the original `babyagi.py` main loop. Watch the queue grow new tasks and re-order itself, then drain to empty.

In [2]:
from collections import deque

OBJECTIVE = "Write a one-paragraph summary of why morning walks boost focus."

# In real BabyAGI each agent below is an OpenAI completion call. Here they are
# deterministic stand-ins so the loop runs offline and reproducibly.

memory = []  # stands in for the vector store of completed (task, result) pairs

def execution_agent(objective, task, context):
    """Perform one task, optionally using a retrieved past result as context."""
    hint = f"  [recalled: {context[0]['task']!r}]" if context else ""
    return f"completed '{task}'{hint}"

# A real task-creation agent invents follow-ups from the result via the LLM.
# We fake it with canned follow-ups keyed by the task just finished.
FOLLOWUPS = {
    "Develop an initial task list": ["Research benefits of morning walks",
                                     "Draft the summary paragraph"],
    "Research benefits of morning walks": ["Cite one supporting study"],
    "Draft the summary paragraph": ["Proofread the summary"],
}

def task_creation_agent(objective, result, task_name, ongoing):
    new = [t for t in FOLLOWUPS.get(task_name, []) if t not in ongoing]
    return [{"task_name": t} for t in new]

def prioritization_agent(objective, task_list):
    """Reorder the queue toward the objective: research -> draft -> proofread."""
    rank = {"Research": 0, "Cite": 1, "Draft": 2, "Proofread": 3}
    keyf = lambda t: next((v for k, v in rank.items()
                           if t["task_name"].startswith(k)), 9)
    return deque(sorted(task_list, key=keyf))

def retrieve_context(query, k=1):
    # Real BabyAGI: top-k by embedding similarity. Here: most recent entries.
    return memory[-k:]

task_list = deque([{"task_name": "Develop an initial task list"}])
MAX_STEPS = 6

for step in range(1, MAX_STEPS + 1):
    if not task_list:
        print(f"\nstep {step}: task list empty — objective complete.")
        break
    task = task_list.popleft()
    context = retrieve_context(OBJECTIVE)
    result = execution_agent(OBJECTIVE, task["task_name"], context)
    memory.append({"task": task["task_name"], "result": result})
    print(f"step {step}: ran {task['task_name']!r}")
    print(f"         -> {result}")

    ongoing = [t["task_name"] for t in task_list]
    for nt in task_creation_agent(OBJECTIVE, result, task["task_name"], ongoing):
        task_list.append(nt)
    task_list = prioritization_agent(OBJECTIVE, task_list)
    print(f"         queue now: {[t['task_name'] for t in task_list]}")

step 1: ran 'Develop an initial task list'
         -> completed 'Develop an initial task list'
         queue now: ['Research benefits of morning walks', 'Draft the summary paragraph']
step 2: ran 'Research benefits of morning walks'
         -> completed 'Research benefits of morning walks'  [recalled: 'Develop an initial task list']
         queue now: ['Cite one supporting study', 'Draft the summary paragraph']
step 3: ran 'Cite one supporting study'
         -> completed 'Cite one supporting study'  [recalled: 'Research benefits of morning walks']
         queue now: ['Draft the summary paragraph']
step 4: ran 'Draft the summary paragraph'
         -> completed 'Draft the summary paragraph'  [recalled: 'Cite one supporting study']
         queue now: ['Proofread the summary']
step 5: ran 'Proofread the summary'
         -> completed 'Proofread the summary'  [recalled: 'Draft the summary paragraph']
         queue now: []

step 6: task list empty — objective complete.


### Example 2 — The memory that makes it work: vector context retrieval

BabyAGI's results would be useless if each task started from scratch. Its **memory** is a vector store: every completed result is embedded, and before running a task the agent retrieves the **top-k most relevant** past results to feed in as context. Here we build a toy vector store using bag-of-words vectors and cosine similarity — no embeddings API needed — to show *why* retrieval beats "just use the last result": the relevant note surfaces even when it isn't the most recent one.

In [3]:
import math
from collections import Counter

def embed(text):
    """Toy 'embedding': a bag-of-words count vector (real BabyAGI uses an LLM)."""
    return Counter(text.lower().split())

def cosine(a, b):
    dot = sum(a[t] * b[t] for t in a)
    na = math.sqrt(sum(v * v for v in a.values()))
    nb = math.sqrt(sum(v * v for v in b.values()))
    return dot / (na * nb) if na and nb else 0.0

class VectorMemory:
    def __init__(self):
        self.items = []                       # list of (text, vector)
    def add(self, text):
        self.items.append((text, embed(text)))
    def query(self, text, k=2):
        q = embed(text)
        ranked = sorted(self.items, key=lambda it: cosine(q, it[1]), reverse=True)
        return [(round(cosine(q, v), 3), t) for t, v in ranked[:k]]

mem = VectorMemory()
for result in [
    "morning walks increase blood flow improving focus and attention",
    "the recommended daily water intake is about two liters",
    "sunlight in the morning helps regulate the circadian rhythm and alertness",
]:
    mem.add(result)

task = "Summarize how morning walks improve focus"
print("Task:", task, "\n")
print("Top-2 retrieved context (score, prior result):")
for score, text in mem.query(task, k=2):
    print(f"  {score}  {text}")
print("\nThe water-intake note is correctly left out — it's irrelevant to the task.")

Task: Summarize how morning walks improve focus 

Top-2 retrieved context (score, prior result):
  0.408  morning walks increase blood flow improving focus and attention
  0.113  sunlight in the morning helps regulate the circadian rhythm and alertness

The water-intake note is correctly left out — it's irrelevant to the task.


### Example 3 — The real thing: `openai` loop & LangChain `BabyAGI`

This is the production shape. It calls a live model and (for the LangChain path) a vector store, so it runs **only** if a library is installed *and* `OPENAI_API_KEY` is set; otherwise it just prints the code you'd write. Note both shapes are the same loop as Example 1 — three prompts around a task queue and a memory.

In [4]:
RAW_LOOP = """
# --- Original BabyAGI, abridged (openai + a vector store) ---
import openai
client = openai.OpenAI()                       # reads OPENAI_API_KEY

def call(prompt):
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content

def execution_agent(objective, task, context):
    return call(f"Objective: {objective}\nContext: {context}\nTask: {task}\nResult:")

def task_creation_agent(objective, result, task, task_list):
    return call(f"Objective: {objective}\nLast result: {result}\n"
                f"Incomplete tasks: {task_list}\nReturn new tasks, one per line.")

def prioritization_agent(objective, task_list):
    return call(f"Reprioritize these tasks for objective '{objective}': {task_list}")

# while task_list: pop -> execute -> store in vector DB -> create -> prioritize
"""

LANGCHAIN = """
# --- Same loop via LangChain's experimental wrapper ---
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.docstore import InMemoryDocstore
from langchain_experimental.autonomous_agents import BabyAGI
import faiss

emb = OpenAIEmbeddings()
index = faiss.IndexFlatL2(len(emb.embed_query("hello")))
store = FAISS(emb, index, InMemoryDocstore({}), {})

baby = BabyAGI.from_llm(llm=OpenAI(temperature=0), vectorstore=store,
                        max_iterations=3)          # <-- always cap iterations!
baby({"objective": "Write a short brief on why morning walks boost focus."})
"""

import importlib.util, os
have_lib = (importlib.util.find_spec("openai") is not None
            or importlib.util.find_spec("langchain_experimental") is not None)

if have_lib and os.getenv("OPENAI_API_KEY"):
    print("Library + key present. In a real session you would run one of the shapes below.")
    print("(Skipped here to keep the notebook fast, offline, and deterministic.)")
else:
    print("openai/langchain not installed or OPENAI_API_KEY unset — code shape only.\n")
    print("# ===== Raw openai loop =====")
    print(RAW_LOOP)
    print("# ===== LangChain BabyAGI =====")
    print(LANGCHAIN)

openai/langchain not installed or OPENAI_API_KEY unset — code shape only.

# ===== Raw openai loop =====

# --- Original BabyAGI, abridged (openai + a vector store) ---
import openai
client = openai.OpenAI()                       # reads OPENAI_API_KEY

def call(prompt):
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content

def execution_agent(objective, task, context):
    return call(f"Objective: {objective}
Context: {context}
Task: {task}
Result:")

def task_creation_agent(objective, result, task, task_list):
    return call(f"Objective: {objective}
Last result: {result}
"
                f"Incomplete tasks: {task_list}
Return new tasks, one per line.")

def prioritization_agent(objective, task_list):
    return call(f"Reprioritize these tasks for objective '{objective}': {task_list}")

# while task_list: pop -> execute -> store in vector DB -> create -> prioritize

# ====

## 6. Gotchas & Pitfalls

- **It does not stop on its own.** The original loop has *no* convergence test — the task-creation agent keeps inventing tasks, so it can run (and bill) forever. **Always** cap iterations (`max_iterations` / a step counter) and/or a budget. This is the single most important thing to know.
- **Task explosion & drift.** Each result can spawn several tasks, so the queue grows faster than it drains, and tasks wander away from the objective. The prioritization agent helps but doesn't fix it; without de-duplication you also get near-identical repeats.
- **Weak grounding.** Out of the box the execution agent just prompts the LLM — no web access, no tools, no verification. Results can be confidently wrong and then become "context" that poisons later tasks. Add tools/retrieval and treat outputs as drafts.
- **Cost scales with the queue.** Every task is 1 execution + 1 creation + 1 prioritization call, each over growing context. A "simple" objective can fan out into dozens of calls quickly.
- **It's a demo, not a framework.** No persistence, retries, observability, or human-in-the-loop. Great for learning and prototyping; don't put the original script in production.
- **Two projects share the name.** "BabyAGI" today might mean the 2023 task loop *or* the 2024 self-building "functionz" reboot. Check which one a tutorial or repo branch refers to before following it.

## 7. When to Use vs Alternatives

| Option | Best for | Trade-offs vs BabyAGI |
|---|---|---|
| **BabyAGI** | Learning the autonomous *plan→act→reflect→re-plan* loop; quick prototypes of open-ended, exploratory goals | No stopping criterion, no tools/guardrails, weak grounding — not production-grade |
| **[[langgraph]]** | Production autonomous agents needing explicit state, branching, cycles, persistence, human-in-the-loop | More to wire up, but you control termination, state, and recovery |
| **[[crewai]]** | Role + task workflows you can describe in plain language with clear structure | Opinionated and bounded by design; less "let it invent its own tasks" |
| **[[autogen]]** | Conversational multi-agent solving (code-write-and-execute, solver+critic) | Multiple agents in dialogue, not one self-directed task list |
| **[[react]] (single agent + tools)** | Goal-directed work where one agent reasons and calls tools step by step | Tool-grounded and easier to bound; no separate task-queue/prioritization machinery |

**Rule of thumb:** use BabyAGI to *understand* autonomous agents and to sketch an idea fast. The moment you care about cost, correctness, tools, or shipping, graduate to LangGraph / CrewAI / AutoGen and keep BabyAGI as the mental model of what they're doing underneath.

## 8. Resources

- **BabyAGI GitHub repository** — https://github.com/yoheinakajima/babyagi
- **Original announcement thread (Yohei Nakajima)** — https://twitter.com/yoheinakajima/status/1640934493489070080
- **"Task-driven Autonomous Agent" write-up** — https://yoheinakajima.com/task-driven-autonomous-agent-utilizing-gpt-4-pinecone-and-langchain-for-diverse-applications/
- **LangChain `BabyAGI` how-to (experimental)** — https://python.langchain.com/docs/use_cases/autonomous_agents/baby_agi/
- **AutoGPT (the sibling autonomous-agent project)** — https://github.com/Significant-Gravitas/AutoGPT
- **Lilian Weng, "LLM Powered Autonomous Agents"** — https://lilianweng.github.io/posts/2023-06-23-agent/